# RQ2 theory allocation probe — CPU only

Validates `geometry -> functional cell mass -> waiting-time cost -> J_p -> pi_p -> q_p` using only frozen development geometry from seeds 0/1/2 and FLOPs metadata. It never reads accuracy and never authorizes training. `p=1` remains the preregistered primary theory policy; other values are mathematical sensitivity checks.

In [ ]:
import os, subprocess, sys, json, shutil
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
print('Commit:', subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip())

## Locate frozen development evidence (no accuracy is loaded)

In [ ]:
import rq2_anchor_placement
RQ2_INPUT = Path('/kaggle/input/notebooks/dyhngg/test-rq2')
assert RQ2_INPUT.exists(), f'Attach RQ2-v1 output: {RQ2_INPUT}'
RQ2_ROOT = rq2_anchor_placement.find_rq2_development_root(RQ2_INPUT, '/kaggle/working/materialized-rq2-theory-probe')
print('Frozen development root:', RQ2_ROOT)

## Run the independent theory probe

In [ ]:
import importlib, rq2_theory_allocation_probe
rq2_theory_allocation_probe = importlib.reload(rq2_theory_allocation_probe)
OUTPUT_DIR = Path('/kaggle/working/theory-allocation-probe')
summary = rq2_theory_allocation_probe.run_theory_probe(RQ2_ROOT, OUTPUT_DIR, waiting_steps=2_000_000)
assert summary['accuracy_used'] is False and summary['test_used'] is False
assert summary['training_authorized'] is False
required = [key for key in summary if key.endswith('_pass')]
assert all(summary[key] for key in required), {key: summary[key] for key in required}
print(json.dumps(summary, indent=2))

In [ ]:
import pandas as pd
from IPython.display import display, Image
cells = pd.read_csv(OUTPUT_DIR/'functional_cells.csv')
family = pd.read_csv(OUTPUT_DIR/'policy_family_summary.csv')
marginals = pd.read_csv(OUTPUT_DIR/'policy_family_marginals.csv')
validation = pd.read_csv(OUTPUT_DIR/'analytic_vs_numeric_validation.csv')
waiting = pd.read_csv(OUTPUT_DIR/'waiting_time_validation.csv')
display(cells)
display(family)
display(marginals.pivot(index='width', columns='p', values='pi'))
display(validation)
print('Maximum waiting-time relative error:', waiting.relative_error.max())
display(Image(filename=str(OUTPUT_DIR/'allocation_family.png')))
display(Image(filename=str(OUTPUT_DIR/'continuous_optimal_density_by_p.png')))
print('THEORY PROBE ONLY — p=1 remains primary; no training is authorized.')

In [ ]:
archive = shutil.make_archive('/kaggle/working/theory-allocation-probe', 'zip', root_dir=OUTPUT_DIR)
print('Download/persist:', archive)